In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
from abc import ABC, abstractmethod
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report

In [2]:
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass

    @abstractmethod
    def backward(self, grad_output, learning_rate):
        pass

class Dense(Layer):
    def __init__(self, input_size, output_size, activation=None):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation

        self.W = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, output_size))

        self.X = None
        self.Z = None

    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b

        if self.activation:
            return self.activation.forward(self.Z)
        return self.Z

    def backward(self, grad_output, learning_rate):
        if self.activation:
            grad = self.activation.backward(self.Z, grad_output)
        else:
            grad = grad_output

        if grad.ndim == 1:
            grad = grad.reshape(-1, 1)

        m = self.X.shape[0]
        dW = (self.X.T @ grad) / m
        db = np.mean(grad, axis=0, keepdims=True)

        grad_input = grad @ self.W.T

        self.W -= learning_rate * dW
        self.b -= learning_rate * db

        return grad_input

class Activation(ABC):
    @abstractmethod
    def forward(self, Z):
        pass

    @abstractmethod
    def backward(self, Z, grad_output):
        pass

class ReLU(Activation):
    def forward(self, Z):
        return np.maximum(0, Z)

    def backward(self, Z, grad_output):
        return grad_output * (Z > 0).astype(float)

class Sigmoid(Activation):
    def forward(self, Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    def backward(self, Z, grad_output):
        A = self.forward(Z)
        return grad_output * A * (1 - A)

class NeuralNetwork:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)
        return self

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
        return grad_output

    def fit(self, X, y, epochs=1000, learning_rate=0.01, batch_size=None, verbose=True):
        if y.ndim == 1:
            y = y.reshape(-1, 1)

        n_samples = X.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            if batch_size:
                for i in range(0, n_samples, batch_size):
                    X_batch = X_shuffled[i:i+batch_size]
                    y_batch = y_shuffled[i:i+batch_size]
                    self._train_batch(X_batch, y_batch, learning_rate)
            else:
                self._train_batch(X_shuffled, y_shuffled, learning_rate)

            if verbose and epoch % 100 == 0:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
            elif epoch == epochs - 1:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                if verbose:
                    print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")

        return self

    def _train_batch(self, X_batch, y_batch, learning_rate):
        y_pred = self.forward(X_batch)
        grad_output = self._loss_gradient(y_batch, y_pred)
        self.backward(grad_output, learning_rate)

    def _compute_loss(self, y_true, y_pred):
        eps = 1e-8
        return -np.mean(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))

    def _loss_gradient(self, y_true, y_pred):
        return y_pred - y_true

    def predict(self, X):
        return self.forward(X)

    def predict_class(self, X, threshold=0.5):
        return (self.predict(X) >= threshold).astype(int)

    def score(self, X, y):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        y_pred = self.predict_class(X)
        return np.mean(y_pred == y)

In [4]:
data = pd.read_csv('train_data.csv')
X = data.drop(['PassengerId', 'Survived'], axis=1).values
y = data['Survived'].values

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = NeuralNetwork()

model.add(Dense(X.shape[1], 32, activation=ReLU()))
model.add(Dense(32, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 16, activation=ReLU()))
model.add(Dense(16, 1, activation=Sigmoid()))

model.fit(X_train, y_train, epochs=1000, learning_rate=0.01, batch_size=16, verbose=True)

print(f"Точность: {model.score(X_test, y_test):.4f}")

Признаков: 15, Объектов: 792
Epoch    0 | Loss: 0.658006
Epoch  100 | Loss: 0.370441
Epoch  200 | Loss: 0.343685
Epoch  300 | Loss: 0.326856
Epoch  400 | Loss: 0.312650
Epoch  500 | Loss: 0.298032
Epoch  600 | Loss: 0.281755
Epoch  700 | Loss: 0.266725
Epoch  800 | Loss: 0.254232
Epoch  900 | Loss: 0.241752
Epoch  999 | Loss: 0.251468
Точность: 0.8176


Нейронная сеть для регрессии

In [5]:
class Layer(ABC):
    @abstractmethod
    def forward(self, X):
        pass

    @abstractmethod
    def backward(self, grad_output, learning_rate):
        pass

class Dense(Layer):
    def __init__(self, input_size, output_size, activation=None):
        self.input_size = input_size
        self.output_size = output_size
        self.activation = activation

        self.W = np.random.randn(input_size, output_size) * np.sqrt(2 / input_size)
        self.b = np.zeros((1, output_size))

        self.X = None
        self.Z = None

    def forward(self, X):
        self.X = X
        self.Z = X @ self.W + self.b

        if self.activation:
            return self.activation.forward(self.Z)
        return self.Z

    def backward(self, grad_output, learning_rate):
        if self.activation:
            grad = self.activation.backward(self.Z, grad_output)
        else:
            grad = grad_output

        if grad.ndim == 1:
            grad = grad.reshape(-1, 1)

        m = self.X.shape[0]
        dW = (self.X.T @ grad) / m
        db = np.mean(grad, axis=0, keepdims=True)

        grad_input = grad @ self.W.T

        self.W -= learning_rate * dW
        self.b -= learning_rate * db

        return grad_input

class Activation(ABC):
    @abstractmethod
    def forward(self, Z):
        pass

    @abstractmethod
    def backward(self, Z, grad_output):
        pass

class ReLU(Activation):
    def forward(self, Z):
        return np.maximum(0, Z)

    def backward(self, Z, grad_output):
        return grad_output * (Z > 0).astype(float)

class Sigmoid(Activation):
    def forward(self, Z):
        return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

    def backward(self, Z, grad_output):
        A = self.forward(Z)
        return grad_output * A * (1 - A)

class Linear(Activation):
    def forward(self, Z):
        return Z

    def backward(self, Z, grad_output):
        return grad_output

class NeuralNetworRegration:
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)
        return self

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, grad_output, learning_rate):
        for layer in reversed(self.layers):
            grad_output = layer.backward(grad_output, learning_rate)
        return grad_output

    def fit(self, X, y, epochs=1000, learning_rate=0.01, batch_size=None, verbose=True):
        if y.ndim == 1:
            y = y.reshape(-1, 1)

        n_samples = X.shape[0]

        for epoch in range(epochs):
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            if batch_size:
                for i in range(0, n_samples, batch_size):
                    X_batch = X_shuffled[i:i+batch_size]
                    y_batch = y_shuffled[i:i+batch_size]
                    self._train_batch(X_batch, y_batch, learning_rate)
            else:
                self._train_batch(X_shuffled, y_shuffled, learning_rate)

            if verbose and epoch % 100 == 0:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")
            elif epoch == epochs - 1:
                y_pred = self.forward(X)
                loss = self._compute_loss(y, y_pred)
                if verbose:
                    print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")

        return self

    def _train_batch(self, X_batch, y_batch, learning_rate):
        y_pred = self.forward(X_batch)
        grad_output = self._loss_gradient(y_batch, y_pred)
        self.backward(grad_output, learning_rate)

    def _compute_loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def _loss_gradient(self, y_true, y_pred):
        return 2 * (y_pred - y_true) / y_true.shape[0]

    def predict(self, X):
        return self.forward(X)


    def score(self, X, y):
        if y.ndim == 1:
            y = y.reshape(-1, 1)
        y_pred = self.predict(X)

        return np.mean(np.abs(y - y_pred)), np.mean((y-y_pred)**2)

In [7]:
data = pd.read_csv('ParisHousing.csv')
X = data.drop('price', axis=1).values
y = data['price'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")


model = NeuralNetworRegration()

model.add(Dense(X.shape[1], 64, activation=ReLU()))
model.add(Dense(64, 128, activation=ReLU()))
model.add(Dense(128, 64, activation=ReLU()))
model.add(Dense(64, 32, activation=ReLU()))
model.add(Dense(32, 1, activation=None))

model.fit(X_train, y_train, epochs=200, learning_rate=0.01, batch_size=32, verbose=True)

y_pred = model.predict(X_test)

score = model.score(X_test, y_test)
print(f"\nMAE: {score[0]:.4f}, MSE: {score[1]:.4f}")



Признаков: 16, Объектов: 10000
Epoch    0 | Loss: 0.769450
Epoch  100 | Loss: 0.057016
Epoch  199 | Loss: 0.035289

MAE: 0.1592, MSE: 0.0408


In [8]:
data = pd.read_csv('ParisHousing.csv')
X = data.drop('price', axis=1).values
y = data['price'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)


scaler_y = StandardScaler()
y = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")


model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),

    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),

    tf.keras.layers.Dense(1)
])


model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae', 'mse']
)

model.fit(X_train, y_train, epochs=20)


Признаков: 16, Объектов: 10000
Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1932 - mae: 0.2854 - mse: 0.1932
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0133 - mae: 0.0915 - mse: 0.0133
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0083 - mae: 0.0721 - mse: 0.0083
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0062 - mae: 0.0625 - mse: 0.0062
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0047 - mae: 0.0549 - mse: 0.0047
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0038 - mae: 0.0491 - mse: 0.0038
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0032 - mae: 0.0449 - mse: 0.0032
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0027 - mae: 0.0413 - mse: 0.0027
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.0021 - mae: 0.0368 - mse: 0.0021
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0018 - mae: 0.0334 - mse: 0.0018
Epoch 11/20
250/25

Тенсер Флоу Классификация

In [99]:
df1 = pd.read_csv("customer_data.csv")
df2 = pd.read_csv("payment_data.csv")
data = pd.merge(df1, df2, on='id', how='inner')
print(data.columns.tolist())
data = data.fillna(data.mean(numeric_only=True))
X = data.drop(['label', 'id', 'update_date', 'report_date'], axis=1).values

y = data['label'].values

scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify= y)

print(f"Признаков: {X.shape[1]}, Объектов: {X.shape[0]}")

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


['label', 'id', 'fea_1', 'fea_2', 'fea_3', 'fea_4', 'fea_5', 'fea_6', 'fea_7', 'fea_8', 'fea_9', 'fea_10', 'fea_11', 'OVD_t1', 'OVD_t2', 'OVD_t3', 'OVD_sum', 'pay_normal', 'prod_code', 'prod_limit', 'update_date', 'new_balance', 'highest_balance', 'report_date']
Признаков: 20, Объектов: 8250


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.8314 - loss: 0.4514 - val_accuracy: 0.8348 - val_loss: 0.4141
Epoch 2/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8352 - loss: 0.4077 - val_accuracy: 0.8470 - val_loss: 0.3951
Epoch 3/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8453 - loss: 0.3730 - val_accuracy: 0.8553 - val_loss: 0.3609
Epoch 4/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8678 - loss: 0.3322 - val_accuracy: 0.8674 - val_loss: 0.3513
Epoch 5/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8839 - loss: 0.2931 - val_accuracy: 0.8682 - val_loss: 0.3156
Epoch 6/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9002 - loss: 0.2582 - val_accuracy: 0.8856 - val_loss: 0.2962
Epoch 7/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9138 - loss: 0.2246 - val_accuracy: 0.8886 - val_loss: 0.2856
Epoch 8/150
165/165 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9205 - loss: 0.2036 - val_acc

In [100]:
print(model.evaluate(X_test, y_test))
y_pred_prob = model.predict(X_test)
print(y_pred_prob)
y_pred = (y_pred_prob > 0.2).astype(int)
print(classification_report(y_test, y_pred))

52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9297 - loss: 0.2547
[0.2547432482242584, 0.9296969771385193]
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
[[3.8136318e-07]
 [2.1557559e-03]
 [7.1197268e-05]
 ...
 [2.0873579e-03]
 [2.9517271e-06]
 [1.8890798e-02]]
              precision    recall  f1-score   support

           0       0.96      0.94      0.95      1373
           1       0.74      0.81      0.77       277

    accuracy                           0.92      1650
   macro avg       0.85      0.88      0.86      1650
weighted avg       0.92      0.92      0.92      1650



In [67]:
df1 = pd.read_csv("customer_data.csv")
df2 = pd.read_csv("payment_data.csv")
result = pd.merge(df1, df2, on='id', how='inner')

In [68]:
result = result.fillna(result.mean(numeric_only=True))

In [17]:
print(X.isna().sum())

fea_1              0
fea_2              0
fea_3              0
fea_4              0
fea_5              0
fea_6              0
fea_7              0
fea_8              0
fea_9              0
fea_10             0
fea_11             0
OVD_t1             0
OVD_t2             0
OVD_t3             0
OVD_sum            0
pay_normal         0
prod_code          0
prod_limit         0
new_balance        0
highest_balance    0
dtype: int64


In [94]:
from sklearn.linear_model import LogisticRegression

X = result.drop(columns=['label', 'id', 'update_date', 'report_date'])
y = result['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression(
    C=1,
    max_iter=1000,
    solver='liblinear',
    class_weight='balanced'
    )

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [95]:
from sklearn.metrics import classification_report
classification_report1 = classification_report(y_test, y_pred)
print(classification_report1)

              precision    recall  f1-score   support

           0       0.89      0.55      0.68      1373
           1       0.23      0.67      0.35       277

    accuracy                           0.57      1650
   macro avg       0.56      0.61      0.52      1650
weighted avg       0.78      0.57      0.63      1650



In [96]:
from sklearn.svm import SVC

svc_model = SVC(
        C=1000,
        kernel='rbf',
        gamma='scale',
        random_state=42)

svc_model.fit(X_train, y_train)
svc_pred = svc_model.predict(X_test)

classification_report2 = classification_report(y_test, svc_pred)
print(classification_report2)

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      1373
           1       0.73      0.65      0.69       277

    accuracy                           0.90      1650
   macro avg       0.83      0.80      0.81      1650
weighted avg       0.90      0.90      0.90      1650

